In [1]:
!pip install qazaq-transliterator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
# import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from qazaq_transliterator import translit
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


def augment_kazakh_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Аугментирует DataFrame, добавляя латинскую транслитерацию
    для всех казахских (kaz) текстов.

    Args:
        df (pd.DataFrame): Исходный DataFrame с колонками 'text' и 'label'.

    Returns:
        pd.DataFrame: Новый DataFrame с оригинальными и аугментированными данными.
    """
    if translit is None:
        raise ImportError("Не удалось импортировать 'translit'. Установите библиотеку.")

    print("--- Начало аугментации данных ---")
    
    # 1. Выбираем только строки с казахским языком
    kaz_samples = df[df['label'] == 'kaz']
    
    if kaz_samples.empty:
        print("Казахские примеры для аугментации не найдены. Возвращаем исходный DataFrame.")
        return df

    print(f"Найдено {len(kaz_samples)} примеров на казахском языке для транслитерации.")

    # 2. Создаем копию этих строк, чтобы не изменять оригинал
    augmented_samples = kaz_samples.copy()

    # 3. Применяем функцию транслитерации к колонке 'text'
    # .apply() проходит по каждой строке и применяет указанную функцию
    augmented_samples['text'] = augmented_samples['text'].apply(
        lambda x: translit(str(x))
    )
    
    # 4. Объединяем исходный DataFrame с новыми, аугментированными данными
    original_len = len(df)
    augmented_df = pd.concat([df, augmented_samples], ignore_index=True)
    
    print("--- Аугментация завершена ---")
    print(f"Размер DataFrame увеличен с {original_len} до {len(augmented_df)} строк.")
    
    return augmented_df
# Загрузка данных
train_df = pd.read_csv("/kaggle/input/faio4-2025/train.csv")
stepan = pd.read_csv('/kaggle/input/faio4-2025/3000.csv')
test_df = pd.read_csv('/kaggle/input/faio4-2025/test.csv')
# test2 = test_df.copy()
# test2['label'] = stepan['label']
# train_df = pd.concat([train_df, test2])

train_df = augment_kazakh_data(train_df)

# Проверка данных
print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")
print(f"Уникальные метки: {train_df['label'].unique()}")
print(f"Распределение классов:\n{train_df['label'].value_counts()}")

# Создание маппинга для меток (если метки текстовые)
label_to_id = {label: idx for idx, label in enumerate(train_df['label'].unique())}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(label_to_id)

# Преобразование меток в числовые значения
train_df['label_id'] = train_df['label'].map(label_to_id)

# Разделение на train и validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label_id'].tolist(),
    test_size=0.005,
    random_state=42,
    stratify=train_df['label_id']
)

# Создание датасетов
def create_dataset(texts, labels=None):
    if labels is not None:
        return Dataset.from_dict({
            'text': texts,
            'labels': labels
        })
    else:
        return Dataset.from_dict({
            'text': texts
        })

# Подготовка датасетов
train_dataset = create_dataset(train_texts, train_labels)
val_dataset = create_dataset(val_texts, val_labels)
test_dataset = create_dataset(test_df['text'].tolist())

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

# Инициализация модели и токенизатора
model_name = "DunnBC22/distilbert-base-multilingual-cased-language_detection_tweets"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Добавление pad токена если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

# Функция токенизации
def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Применение токенизации
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Функция для вычисления метрик
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Настройка параметров обучения
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.02,
    warmup_steps=500,
    logging_dir='./',
    logging_steps=50,
    metric_for_best_model='f1',
    greater_is_better=True,
    push_to_hub=False,
    fp16=True,
    report_to='none',  # Отключить wandb/tensorboard
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Обучение модели
print("Начинаем обучение...")
trainer.train()

# Оценка на валидационной выборке
print("\nОценка модели на валидационной выборке:")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# Сохранение модели
trainer.save_model('./final_model')
tokenizer.save_pretrained('./final_model')

# Сохранение маппинга меток
import json
with open('./final_model/label_mapping.json', 'w') as f:
    json.dump({
        'label_to_id': label_to_id,
        'id_to_label': id_to_label
    }, f)

# Предсказание на тестовой выборке
print("\nГенерация предсказаний для тестовой выборки...")
test_predictions = trainer.predict(tokenized_datasets['test'])
predicted_labels = np.argmax(test_predictions.predictions, axis=1)

# Преобразование обратно в исходные метки
predicted_labels_text = [id_to_label[label_id] for label_id in predicted_labels]

# Создание submission файла
submission = pd.DataFrame({
    'id': test_df['id'],
    'label': predicted_labels_text
})

submission.to_csv('submission.csv', index=False)
print(f"\nФайл submission.csv создан. Количество предсказаний: {len(submission)}")

# Функция для предсказания новых текстов
def predict_texts(texts, model_path='./final_model'):
    """
    Функция для предсказания классов новых текстов
    """
    # Загрузка модели и токенизатора
    loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Загрузка маппинга меток
    with open(f'{model_path}/label_mapping.json', 'r') as f:
        mappings = json.load(f)
        id_to_label = {int(k): v for k, v in mappings['id_to_label'].items()}
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loaded_model.to(device)
    loaded_model.eval()
    
    predictions = []
    probabilities = []
    
    with torch.no_grad():
        for text in texts:
            inputs = loaded_tokenizer(
                text, 
                return_tensors='pt',
                truncation=True,
                padding=True,
                max_length=128
            ).to(device)
            
            outputs = loaded_model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            pred = torch.argmax(logits, dim=-1)
            
            predictions.append(id_to_label[pred.cpu().numpy()[0]])
            probabilities.append(probs.cpu().numpy()[0])
    
    return predictions, probabilities

# Пример использования функции предсказания
if __name__ == "__main__":

    sample_texts = test_df['text'].head(5).tolist()
    predictions, probs = predict_texts(sample_texts)
    
    print("\nПримеры предсказаний:")
    for i, (text, pred, prob) in enumerate(zip(sample_texts, predictions, probs)):
        print(f"\nТекст {i+1}: {text[:100]}...")
        print(f"Предсказание: {pred}")
        print(f"Вероятности: {dict(zip(label_to_id.keys(), prob))}")

--- Начало аугментации данных ---
Найдено 1013 примеров на казахском языке для транслитерации.
--- Аугментация завершена ---
Размер DataFrame увеличен с 3863 до 4876 строк.
Размер обучающей выборки: 4876
Размер тестовой выборки: 15315
Уникальные метки: ['eng' 'kaz' 'ru']
Распределение классов:
label
kaz    2026
eng    1670
ru     1180
Name: count, dtype: int64


Map:   0%|          | 0/4851 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/15315 [00:00<?, ? examples/s]

/tmp/ipykernel_100/3320443734.py:182: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.082200,0.044980,0.960000,0.960486,0.965714,0.960000
2,0.024200,0.000112,1.000000,1.000000,1.000000,1.000000
3,0.028600,0.000000,1.000000,1.000000,1.000000,1.000000


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Оценка модели на валидационной выборке:


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


eval_loss: 0.0000
eval_accuracy: 1.0000
eval_f1: 1.0000
eval_precision: 1.0000
eval_recall: 1.0000
eval_runtime: 0.1476
eval_samples_per_second: 169.3440
eval_steps_per_second: 6.7740
epoch: 3.0000

Генерация предсказаний для тестовой выборки...


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
from scipy.sparse import hstack
import warnings
# Для проверки наличия GPU
try:
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    USE_GPU = False


warnings.filterwarnings('ignore')

# --- 1. Загрузка данных ---
print("1. Загрузка данных...")
train_df = pd.read_csv("/kaggle/input/foai-train-data/train.csv")
test_df = pd.read_csv("/kaggle/input/foai-train-data/test.csv")
full_df = pd.concat([train_df['text'], test_df['text']], axis=0).reset_index(drop=True)


# --- 3. Векторизация текста ---
print("3. Векторизация текста (TF-IDF)...")
word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=1000, sublinear_tf=True)
char_vectorizer = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 5), max_features=2000, sublinear_tf=True)

print("   - Обучение векторизаторов...")
word_vectorizer.fit(full_df.astype(str))
char_vectorizer.fit(full_df.astype(str))

print("   - Трансформация данных...")
train_word_features = word_vectorizer.transform(train_df['text'].astype(str))
test_word_features = word_vectorizer.transform(test_df['text'].astype(str))
train_char_features = char_vectorizer.transform(train_df['text'].astype(str))
test_char_features = char_vectorizer.transform(test_df['text'].astype(str))

# --- 4. Объединение всех признаков ---

# --- 5. Подготовка к обучению и разделение данных ---
print("5. Подготовка данных для обучения и валидации...")
label_encoder = LabelEncoder()
y_train_full_encoded = label_encoder.fit_transform(train_df['label'])

# Разделяем полный тренировочный набор на обучающую и валидационную выборки
# test_size=0.2 означает, что 20% данных пойдет на валидацию
# stratify=y_train_full_encoded гарантирует, что в обеих выборках будет одинаковое соотношение классов
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_full, 
    y_train_full_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train_full_encoded
)

print(f"Размер обучающей выборки (split): {X_train_split.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")

# --- 6. Обучение модели с валидацией и ранней остановкой ---
print("\n6. Обучение модели с использованием валидационной выборки...")

model = CatBoostClassifier(
    iterations=3000,  # Ставим большое число, early_stopping сам найдет оптимум
    learning_rate=0.05,
    depth=7,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro', # Оптимизируем под Macro F1
    verbose=200,
    random_seed=42,
    task_type='GPU' if USE_GPU else 'CPU',
    early_stopping_rounds=100 # Остановить обучение, если Macro F1 на валидации не улучшается 100 итераций
)

model.fit(
    X_train_split, y_train_split,
    eval_set=(X_val, y_val),
    use_best_model=True # Модель автоматически вернется к лучшей итерации
)

# Получаем оптимальное количество итераций
best_iteration = model.get_best_iteration()
print(f"\nОптимальное количество итераций найдено: {best_iteration}")
print(f"Лучший Macro F1 на валидации: {model.get_best_score()['validation']['TotalF1:average=Macro']:.5f}")


# --- 7. Переобучение финальной модели на всех данных ---
print("\n7. Переобучение финальной модели на всех тренировочных данных...")

# Создаем новую модель с теми же параметрами, но обучаем ее на всех данных
# Устанавливаем количество итераций равным найденному оптимуму
final_model = CatBoostClassifier(
    learning_rate=0.05,
    depth=7,
    loss_function='MultiClass',
    verbose=200,
    random_seed=42,
    task_type='GPU' if USE_GPU else 'CPU'
)

# Обучаем на X_train_full (100% данных)
final_model.fit(X_train_full, y_train_full_encoded)


# --- 8. Предсказание и сохранение результата ---
print("\n8. Предсказание и формирование submission файла...")
predictions_ids = final_model.predict(X_test_full).flatten()
predictions_labels = label_encoder.inverse_transform(predictions_ids)

submission = pd.DataFrame({'id': test_df.index, 'label': predictions_labels})
submission.to_csv('solution.csv', index=False)

print("\nФайл 'solution.csv' успешно создан!")
print("Пример содержимого:")
print(submission.head())

In [ ]:
!pip install fasttext

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/foai-train-data/test.csv')
arr_len = []

for i in df.values:
    arr_len.append(len(i[1]))
print(min(arr_len))